In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import jax
jax.clear_caches()
print(jax.__version__)
print(jax.devices())

0.6.1
[CudaDevice(id=0), CudaDevice(id=1), CudaDevice(id=2), CudaDevice(id=3)]


In [2]:
from typing import Tuple, List
import math
from operator import add
import numpy as onp  
import jax.numpy as jnp
from jax import grad, jit, value_and_grad, random, lax, device_get
from functools import partial
from dataclasses import dataclass
from jax.tree_util import register_dataclass
from jax.tree_util import tree_map
from typing import Any

@register_dataclass
@dataclass
class LayerNormParams:
    gamma: jnp.ndarray
    beta:  jnp.ndarray
 
@register_dataclass
@dataclass
class AttnParams:
    w:     jnp.ndarray        # your attn_w   (d,)
    bn:    LayerNormParams    # BN over attention output (N, d)

@register_dataclass
@dataclass
class FfnParams:
    in_W: jnp.ndarray       # (m, d)
    out_W: jnp.ndarray      # (d, m)
    bn:   LayerNormParams   # BN over pre_h (N, m)

@register_dataclass
@dataclass
class EmbeddingParams:
    x:    jnp.ndarray          # (V, d)
    y:    jnp.ndarray          # (V, d)
    trig: jnp.ndarray       # (d, )


@register_dataclass
@dataclass
class ModelParams:
    attn: AttnParams
    ffn:  FfnParams

 

@register_dataclass
@dataclass
class AdamState:
    m: Any   # pytree, same structure as params
    v: Any   # pytree, same structure as params
    t: jnp.ndarray  # scalar int


def adam_init(params: ModelParams) -> AdamState:
    m = tree_map(jnp.zeros_like, params)
    v = tree_map(jnp.zeros_like, params)
    t = jnp.array(0, dtype=jnp.int32)
    return AdamState(m=m, v=v, t=t)


def adam_update(
    params: ModelParams,
    grads:  ModelParams,
    state:  AdamState,
    lr: float = 1e-3,
    beta1: float = 0.9,
    beta2: float = 0.999,
    eps: float = 1e-8,
) -> tuple[ModelParams, AdamState]:
    """Single Adam step on a pytree of params."""

    t = state.t + 1

    m_new = tree_map(
        lambda m, g: beta1 * m + (1.0 - beta1) * g,
        state.m, grads
    )
    
    v_new = tree_map(
        lambda v, g: beta2 * v + (1.0 - beta2) * (g * g),
        state.v, grads
    )


    # Bias correction scalars
    m_hat_factor = 1.0 / (1.0 - beta1**t)
    v_hat_factor = 1.0 / (1.0 - beta2**t)

    def apply_update(param, m, v):
        m_hat = m * m_hat_factor
        v_hat = v * v_hat_factor
        step = lr * m_hat / (jnp.sqrt(v_hat) + eps)
        return param - step

    params_new = jax.tree_util.tree_map(
        apply_update, params, m_new, v_new
    )

    new_state = AdamState(m=m_new, v=v_new, t=t)
    return params_new, new_state





In [3]:
def layer_norm(x, bn: LayerNormParams,  eps=1e-5):
    """
    x:     (..., d)
    gamma: (d,)
    beta:  (d,)
    Normalize over the last dimension.
    """
    mean = jnp.mean(x, axis=-1, keepdims=True)         # (..., 1)
    var  = jnp.var(x, axis=-1, keepdims=True)          # (..., 1)

    x_hat = (x - mean) / jnp.sqrt(var + eps)           # (..., d)
    return bn.gamma * x_hat + bn.beta



def activation_function(x):
    return  0.7 * (x ** 2 - 1)/jnp.sqrt(2) + 0.3 * (x ** 3 - 3 * x)/jnp.sqrt(6)

def make_perm_params(key, V: int):
    # 1) sample the shift b uniformly
    key, k1 = random.split(key)
    b = int(random.randint(k1, (), 0, V))

    # 2) sample a until gcd(a, V) == 1
    while True:
        key, k2 = random.split(key)
        a = int(random.randint(k2, (), 1, V))
        if math.gcd(a, V) == 1:
            break

    return   a, b

def generate_data(key, V=50, N=300, L=1):
    key, *subkeys = jax.random.split(key, 6)
    perm_a, perm_b = make_perm_params(subkeys[0], V)

    latent_train = jax.random.choice(subkeys[1], V, shape=(N,))
    targets = (perm_a * latent_train + perm_b) % V 

    latent_test = jnp.arange(V)
    test_targets = (perm_a * latent_test + perm_b) % V 

    if L == 1:
        inputs = latent_train[:, None]
        test_inputs = latent_test[:, None]
    else:
        noise_tokens = random.choice(subkeys[3], V, shape=(N, L - 1))
        inputs = jnp.concatenate((latent_train[:, None], noise_tokens), axis=1)

        noise_tokens = random.choice(subkeys[4], V, shape=(V, L - 1))
        test_inputs = jnp.concatenate((latent_test[:, None], noise_tokens), axis=1)

    return (perm_a, perm_b), inputs, targets, test_inputs, test_targets, key


def generate_embedding(key, V=50, d=100, snr = 0.1):
    key, *subkeys = jax.random.split(key,4)
    embedding_x = (1 / jnp.sqrt(d)) * jax.random.normal(subkeys[0], (V, d))
 
     
    trigger = (snr / jnp.sqrt(d)) * jax.random.normal(subkeys[1], (d, ))
    
    embedding_y = (1 / jnp.sqrt(d)) * jax.random.normal(subkeys[2], (V, d))
    return embedding_x, trigger, embedding_y, key

def attention_layer(params_attn, emb, trigger):
    emb_trig         = emb.at[:, 0, :].add(trigger)
    pre_activation   = jnp.einsum('ntd,d->nt', emb_trig, params_attn.w)   # (N, L, d) -> (N,L)
    pre_activation_ln = layer_norm( pre_activation, params_attn.bn,  eps=1e-5)
    attention_scores = jax.nn.softmax(pre_activation_ln , axis=1) 
    o = jnp.einsum('nt,ntd->nd', attention_scores, emb)   # (N,d)
    return  o


def linear_layer(params_ffn, x_repr):
    return  x_repr @  params_ffn.in_W.T     # (N, d)



def non_linear_layer(f_W, in_W, inputs, embedding_y):  
    ## TODO: Complete here
    h = activation_function(in_W @ inputs.T)  
    o = f_W @ h
    logits = embedding_y @ o
    return jax.nn.softmax(logits.T, axis=1)

 
def logistic_loss_expired(h_ln, emb_y, labels):
    true_logits = jnp.sum(h_ln * emb_y[labels], axis=-1) 
    logZ        = _logsumexp_chunked(h_ln, emb_y)  
    return jnp.mean(-true_logits + logZ)  

def logistic_loss(probs,  labels):
    batch_size = probs.shape[0]
    correct_probs = probs[jnp.arange(batch_size), labels]
    cross_entropy = -jnp.mean(jnp.log(correct_probs + 1e-12))
    return cross_entropy 



In [4]:
def compute_acc(
    params:      ModelParams,
    emb_x:       jnp.ndarray,
    emb_y:       jnp.ndarray,
    trig:        jnp.ndarray,
    labels:      jnp.ndarray
) -> Tuple[jnp.ndarray, jnp.ndarray]:
    
     # unpack
    attn = params.attn
    ffn  = params.ffn

 
    features    = attention_layer(attn, emb_x, trig)
    
     
    h    = linear_layer(ffn,  features)
    h_ln = layer_norm( h, ffn.bn,  eps=1e-5)
    
    logits = h_ln @  emb_y 
    probs  = jax.nn.softmax(logits)      
    preds = jnp.argmax(probs, axis=1)
    acc   = jnp.mean(preds == labels) 
 
    return  acc 



def compute_loss(
    params:      ModelParams,
    emb_x:       jnp.ndarray,
    emb_y:       jnp.ndarray,
    trig:        jnp.ndarray,
    labels:      jnp.ndarray,
    lam:         float = 0.0 
) -> Tuple[jnp.ndarray, jnp.ndarray]:
    
     # unpack
    attn = params.attn
    ffn  = params.ffn

 
    features    = attention_layer(attn, emb_x, trig)
    
     
    h    = linear_layer(ffn,  features)
    h_ln = layer_norm( h, ffn.bn,  eps=1e-5)
    
    logits = h_ln @  emb_y 
    probs  = jax.nn.softmax(logits)   
    loss   = logistic_loss(probs, labels)
    reg_term   = 0.5 * lam * jnp.sum(ffn.in_W ** 2)
    return loss + reg_term 


_grad_params   = jit(value_and_grad(compute_loss))



def make_multi_device_step(
    embeds:         EmbeddingParams,
    micro_batch:    int,  
    lam:            float = 0.0 
):
 
    devices = jax.local_devices()
    D = len(devices)

    emb_x = embeds.x
    emb_y = embeds.y
    trig  = embeds.trig
    
    @partial(jax.pmap,
             axis_name="devices",
             in_axes=(0,    # key per-device
                      None, # params broadcast
                      0,    # ids_shard
                      0))  
    def _step(
        key:          jnp.ndarray,
        params:       ModelParams,  
        ids_shard:    jnp.ndarray,
        labels_shard: jnp.ndarray 
    ):

        # _, g_local   = _grad_params(params,  emb_x[ids_shard], emb_y.T, trig, labels_shard, lam)
        N  = ids_shard.shape[0] 

        B =  micro_batch
        P     = (N + B - 1) // B
        total = B * P
        idx     = jnp.arange(total) % N 
 
        batches = idx.reshape((B, P))
        # compute each device’s local chunked gradient

        g0 = tree_map( jnp.zeros_like, params)

        def body(g_acc, batch_idx):
            
            train_ind_b = ids_shard[batch_idx]
            train_lbls_b= labels_shard[batch_idx]
            _, g   = _grad_params(params,  emb_x[train_ind_b], emb_y.T, trig, train_lbls_b, lam)

            gres =  tree_map(add, g_acc, g)
            return  gres, None

        g_total, _ = lax.scan(body, g0, batches)
        g_local    = tree_map(lambda x: x / B, g_total)

        # # synchronous all‐reduce (mean) across devices
        g_full = lax.pmean(g_local, axis_name="devices")

        # advance RNG
        new_key = random.split(key, 1)[0]
        return g_full, new_key

    return _step

 
def grad_compute(
    key, params, ids, labels,
    multi_step=None,
):
    """
    Compute the gradient of the loss w.r.t. params using multi-GPU data parallelism.

    multi_step: the pmapped function returned by make_multi_device_step,
                built once per run (see run_one below).
    """


    devices = jax.local_devices()
    D = len(devices)

    # ids: (N, T), labels: (N,)
    N, L = ids.shape
    # N_pad = ((N + D - 1) // D) * D

    # if N_pad > N:
    #     pad = N_pad - N
    #     idx = jnp.arange(pad) % N   # wrap around safely even when N < D
    #     ids    = jnp.concatenate([ids,    ids[idx]],    axis=0)
    #     labels = jnp.concatenate([labels, labels[idx]], axis=0)
    #     N = N_pad

    # # shard across devices: (D, shard, ...)
    shard = N // D
    ids_shard    = ids.reshape((D, shard, L))
    labels_shard = labels.reshape((D, shard))

    # split RNG for each device
    keys = random.split(key, D)

    # call the pre-built pmapped step
    g_accs, new_keys = multi_step(
        keys,
        params,
        ids_shard,
        labels_shard,
    )

    # gradients are identical on all devices after pmean, take device 0
    g_res = tree_map(lambda x: x[0], g_accs)
    return g_res, new_keys[0]




def _test_metrics(embeds):      
    emb_x = embeds.x
    emb_y = embeds.y
    trig  = embeds.trig


    def test_accuracy(params,test_ids, test_labels):
        return compute_acc(params, emb_x[test_ids] , emb_y.T, trig, test_labels)
    
    return test_accuracy
 

def test_metrics(key, params,  embeds, ids, lbls,  partition_size = 128):
    N   = ids.shape[0]
    idx = jnp.arange(N)
 
    B     = int(jnp.ceil(N / partition_size))
    total = B * partition_size
    pad   = total - N         # how many extra we need
 
    if pad > 0:
        key, sub = jax.random.split(key)
        # sample with replacement
        extra = jax.random.choice(sub, N, shape=(pad,), replace=True)
        idx   = jnp.concatenate([idx, extra], axis=0)

    test_accuracy = _test_metrics(embeds)
    batches = idx.reshape((B, partition_size))

    def test_body(carry, batch_ids):
            acc = carry
            test_ids_b =  ids[batch_ids]
            test_lbl_b =  lbls[batch_ids]
        
            at = test_accuracy(params, test_ids_b, test_lbl_b)
            acc += at

            return acc, None

    acc, _ = lax.scan(test_body, 0.0, batches)

    return  acc/B, key


In [5]:
if __name__ == "__main__":
    
    def run_one(key, V: int = 50, N: int = 250, d: int = 40, L: int = 1, num_epochs: int = 1500, lr_alg: float = 0.1, iter_per_epoch: int = 2) -> float:

        # Initialize parameters
        bn_attn =  LayerNormParams(
            gamma=jnp.ones((L,)),
            beta=jnp.zeros((L,)) )

        bn_ffn =  LayerNormParams(
            gamma=jnp.ones((d,)),
            beta=jnp.zeros((d,)) )

        
        params = ModelParams(
                    attn=AttnParams(
                        w= jnp.zeros(d),
                        bn = bn_attn ),
                    ffn=FfnParams(
                        in_W  = jnp.zeros((d, d)),
                        out_W = 0.0,
                        bn = bn_ffn) )

 

        emb_x, trigger, emb_y, key = generate_embedding(key, V, d, snr = 0.2) # 0.1 if small signal  
        embeds = EmbeddingParams( x =  emb_x, y = emb_y, trig = trigger)
        perm, train_ids, train_lbls, test_ids, test_lbls, key = generate_data(key, V, N, L)
        # assume `adam` is part of your carry/state and initialized with init_adam_state(...
        adam = adam_init(params)

        mb = 1
        partition_size = 1024
        if jnp.log2(V) > 10.1 and jnp.log2(d) > 4.1:
            mb = 32
            partition_size = 128

        multi_step = make_multi_device_step(
            embeds=embeds,
            micro_batch = mb,
            lam= 5e-15,
        )
 
        
        lr, beta1, beta2, eps =  lr_alg,  0.9, 0.999  , 1e-8    
        batch_size = int(N/iter_per_epoch)
        # batch_size = 2048
        acc_res = jnp.zeros(  num_epochs + 1)
        # print("L:",L,"remainder", N % batch_size)
        
        # First step
        acc_res = acc_res.at[0].set(0)

 
        def one_step(carry):
            key, params, train_b, label_b, adam = carry
        
            # Multi-GPU gradient using pre-built pmapped step
            gp, key = grad_compute(
                key,
                params,
                ids=train_b,
                labels=label_b,
                multi_step=multi_step,    
            )

            # Adam update on host
            params_new, adam = adam_update(
                params,
                gp,
                adam,
                lr=lr,
                beta1=beta1,
                beta2=beta2,
                eps=eps,
            )
            return (key, params_new, adam)


        # ——— SGD training loop: shuffle once/epoch, iterate minibatches ———
        B = batch_size       # minibatch size
        N = train_ids.shape[0]

        if N <= B:
            B = N
            steps_per_epoch = 1
        else:
            steps_per_epoch = (N + B - 1) // B
            
        num_epochs = num_epochs               # set this upstream
        # early-stopping bookkeeping you already had
        ch = 0
        idx = 1
        global_step = 0
        early_stopped = False
        key, params_curr= key, params

        for epoch in range(num_epochs):
            # Shuffle once per epoch
            key, sub = jax.random.split(key)
            perm     = jax.random.permutation(sub, N)
            ids_shuf = train_ids[perm]
            lbl_shuf = train_lbls[perm]
            # print("Epoch:", epoch)
            for s in range(steps_per_epoch):
                start   = s * B 
                end     = jnp.minimum(start + B, N)
                train_b = ids_shuf[start:end]
                label_b = lbl_shuf[start:end]
        
                # One SGD step on this minibatch
                (key,  params_curr, adam) = one_step((key,  params_curr, train_b, label_b, adam))

            # Metrics / early stop every `freq` steps (same logic as before)
            acc, key = test_metrics(key, params_curr,  embeds, test_ids, test_lbls, partition_size)
            acc_res = acc_res.at[idx].set(acc)
            # print("Epoch:", epoch, "Accuracy:", acc)
            idx = idx + 1

  
 
        return  params_curr, acc_res,  key 


    # print("mert")
    seed     = 12
    key      = jax.random.PRNGKey(seed)
    devices  = jax.local_devices()
    D        = len(devices)
    iter_per_epoch = 2
    print("Device size:", D, "Iteration per epoch:", iter_per_epoch)
    # batch_size = 2048
    # print("Device size:", D, "Batch size:", batch_size)
      
    Vlis      = (2.0 **  jnp.linspace(9.0,13.0, num= 20) ).astype(int)  
    dlis      = jnp.ceil(2.0 ** jnp.linspace(6.0,12.0, num= 60)).astype(int) 
    Llis      = [None]
    Nlis      = [None]
    repeat    = 1
    lr        = 0.005
    num_epoch = 25

    num_runs  = len(Vlis) * len(Nlis) *  len(Llis) *   len(dlis) * repeat
    
    key, *subkeys  = random.split(key, num_runs + 1)
    idx      = 0

    Vlis          = Vlis[-5:] 
    eff_num_runs  = len(Vlis) * len(Nlis) *  len(Llis) *   len(dlis) * repeat
    subkeys       = subkeys[-eff_num_runs:]
    num_runs      = eff_num_runs

    Deff =  (D* iter_per_epoch)
    Ns   =   0.5 *  ( Vlis ** 1.5) #    Vlis *  (jnp.log(Vlis)  + 5.1 )
    Ns   =  (  jnp.round(  Ns   /  Deff )  * Deff   ).astype(int)
    Ls   =  (  Vlis / 8 ).astype(int)
    print("Vlis", Vlis)
    print("Nlis", Ns)
    print("Llis", Ls)

    
    shape        = (len(Vlis), len(Nlis), len(Llis), len(dlis), repeat, num_epoch + 1)
    accuracy_vec = jnp.zeros(shape, dtype=jnp.float32)
    repeat_vec   =  jnp.zeros(repeat, dtype=jnp.float32)
 

    signal_vec =  jnp.zeros(( len(dlis),repeat), dtype=jnp.float32)
    noise_vec  =  jnp.zeros(( len(dlis),repeat), dtype=jnp.float32)
    
    
    
    for i, Vcurrent in enumerate(Vlis):
        for j, Nval in enumerate(Nlis):
            Ncurrent = 0.5 *  ( Vcurrent ** 1.5) #  Vcurrent *  (jnp.log(Vcurrent)  + 5.1 )
            # Ncurrent =  ( jnp.round(  Ncurrent   /   batch_size ) * batch_size   ).astype(int)
            Ncurrent =  (  jnp.round(  Ncurrent  /  Deff )  * Deff   ).astype(int)
            for l, Lval in enumerate(Llis):
                if Lval is None:
                    Lcurrent = int(  Vcurrent / 8 )
  
                for k, dcurrent in enumerate(dlis):
                    for r in range(repeat):
                    
                        subkey = subkeys[idx]
                        param_final, acc_res, _  = run_one(
                            subkey, Vcurrent, Ncurrent,
                            dcurrent, Lcurrent,
                            num_epoch, lr, iter_per_epoch
                        )
                        repeat_vec = repeat_vec.at[r].set(float(acc_res[-1]))


                        print(f"[{idx+1}/{num_runs}] V={Vcurrent}, N={Ncurrent}, "
                          f"L={Lcurrent}, d={dcurrent}, r={r}", "Accuracy", acc_res[1:])

                        idx += 1
                        jax.clear_caches()
                        
                        accuracy_vec = accuracy_vec.at[i, j, l, k, r, :].set(acc_res )

                    
                    
                    # if  jnp.median(repeat_vec) > 0.75:
                    #     last_res = accuracy_vec[i, j, l, k, :, :]              # (A, B)
                    #     accuracy_vec = accuracy_vec.at[i, j, l, k:, :, :].set(
                    #         jnp.broadcast_to(last_res, accuracy_vec[i, j, l, k:, :, :].shape)
                    #     )
                    #     print("Skipping d =",dlis[k:])
                    #     idx += len(dlis[k:]) * repeat - repeat
                    #     break
                   
                        
  
                    
                    host_acc =  device_get(accuracy_vec)
                    onp.savez_compressed(
                        "checkpoint_linear_accuracy_vec_gd_LVover8_BNover2_epoch25_more_data_rem.npz", 
                        accuracy_vec=host_acc,
                        Vlis=Vlis, Nlis=Ns, dlis=dlis, Llis=Ls,
                        seed=seed, lr=lr,
                        device_size = D,
                        part_size = iter_per_epoch
                        # batch_size = batch_size
                    )

    print("Done")

  

Device size: 4 Iteration per epoch: 2


/mnt/sw/nix/store/71ksmx7k6xy3v9ksfkv5mp5kxxp64pd6-python-3.10.13-view/lib/python3.10/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/mnt/sw/nix/store/71ksmx7k6xy3v9ksfkv5mp5kxxp64pd6-python-3.10.13-view/lib/python3.10/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)


Vlis [4569 5287 6118 7079 8192]
Nlis [154416 192216 239264 297800 370728]
Llis [ 571  660  764  884 1024]


2026-02-23 16:15:10.963069: E external/xla/xla/service/rendezvous.cc:92] [id=2] This thread has been waiting for `initialize clique for rank 2; clique=devices=[0,1,2,3]; stream=0; groups=[[0,1,2,3]]; root_device=-1; num_local_participants=4; run_id=-492526811` for 10 seconds and may be stuck. All 4 threads joined the rendezvous, however the leader has not marked the rendezvous as completed. Leader can be deadlocked inside the rendezvous callback.
2026-02-23 16:15:10.963094: E external/xla/xla/service/rendezvous.cc:92] [id=1] This thread has been waiting for `initialize clique for rank 3; clique=devices=[0,1,2,3]; stream=0; groups=[[0,1,2,3]]; root_device=-1; num_local_participants=4; run_id=-492526811` for 10 seconds and may be stuck. All 4 threads joined the rendezvous, however the leader has not marked the rendezvous as completed. Leader can be deadlocked inside the rendezvous callback.
2026-02-23 16:15:10.963147: E external/xla/xla/service/rendezvous.cc:92] [id=0] This thread has be

[1/300] V=4569, N=154416, L=571, d=64, r=0 Accuracy [0.00021701 0.00065104 0.00021701 0.         0.00021701 0.00021701
 0.         0.00021701 0.         0.00021701 0.00021701 0.00043403
 0.00086806 0.00021701 0.00065104 0.00065104 0.00065104 0.00065104
 0.00043403 0.00043403 0.00043403 0.00043403 0.00043403 0.00065104
 0.00108507]
[2/300] V=4569, N=154416, L=571, d=69, r=0 Accuracy [0.00021701 0.         0.00021701 0.00021701 0.00021701 0.00065104
 0.00021701 0.         0.         0.00043403 0.00043403 0.
 0.00086806 0.00065104 0.00043403 0.00043403 0.00021701 0.00086806
 0.0015191  0.00065104 0.0015191  0.00303819 0.00368924 0.00542535
 0.00716146]
[3/300] V=4569, N=154416, L=571, d=74, r=0 Accuracy [0.00021701 0.00021701 0.         0.00021701 0.00065104 0.00021701
 0.00021701 0.00021701 0.00021701 0.00086806 0.00043403 0.00021701
 0.00021701 0.00043403 0.00043403 0.00021701 0.00043403 0.00043403
 0.00065104 0.00065104 0.00108507 0.0015191  0.00195312 0.00390625
 0.00651042]
[4/300] V

In [6]:
print(accuracy_vec)

[[[[[[0.0000000e+00 2.1701389e-04 6.5104169e-04 ... 4.3402778e-04
      6.5104169e-04 1.0850695e-03]]

    [[0.0000000e+00 2.1701389e-04 0.0000000e+00 ... 3.6892362e-03
      5.4253475e-03 7.1614585e-03]]

    [[0.0000000e+00 2.1701389e-04 2.1701389e-04 ... 1.9531250e-03
      3.9062500e-03 6.5104165e-03]]

    ...

    [[0.0000000e+00 5.3146702e-01 1.0000000e+00 ... 1.0000000e+00
      1.0000000e+00 1.0000000e+00]]

    [[0.0000000e+00 5.6944448e-01 1.0000000e+00 ... 1.0000000e+00
      1.0000000e+00 1.0000000e+00]]

    [[0.0000000e+00 7.0182294e-01 1.0000000e+00 ... 1.0000000e+00
      1.0000000e+00 1.0000000e+00]]]]]




 [[[[[0.0000000e+00 1.8601191e-04 0.0000000e+00 ... 5.5803574e-04
      3.7202382e-04 5.5803574e-04]]

    [[0.0000000e+00 1.8601191e-04 0.0000000e+00 ... 3.7202382e-04
      1.8601191e-04 7.4404763e-04]]

    [[0.0000000e+00 3.7202382e-04 7.4404763e-04 ... 1.1160715e-03
      1.8601191e-03 3.7202381e-03]]

    ...

    [[0.0000000e+00 2.7343750e-01 1.0000000e+00 .